In [1]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_1434_Wazirpur_Delhi_DPCC_1Day.csv")

In [4]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,199.22,336.47,15.02,60.92,75.95,29.70,9.35,1.02,8.79,...,NaN,10.39,77.89,1.16,78.01,0.0,0.0,39.18,978.26,NaN
1,2024-01-02,193.07,329.67,23.40,62.58,85.93,25.21,12.11,1.21,8.84,...,NaN,10.22,75.17,1.12,75.89,0.0,0.0,56.21,978.37,NaN
2,2024-01-03,225.34,351.88,32.58,66.10,98.89,32.93,12.18,1.75,3.35,...,NaN,9.64,85.57,1.03,82.43,0.0,0.0,29.76,978.49,NaN
3,2024-01-04,259.58,415.72,48.20,74.59,95.02,26.08,19.48,1.03,3.44,...,NaN,9.80,85.20,0.88,139.63,0.0,0.0,16.64,978.38,NaN
4,2024-01-05,194.77,333.77,53.25,114.85,104.38,21.22,20.20,1.14,5.08,...,NaN,10.46,88.32,1.15,228.93,0.0,0.0,11.32,978.22,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,266.12,362.06,27.52,88.81,69.60,87.18,6.83,2.52,51.74,...,NaN,13.51,77.71,1.01,269.26,0.0,0.0,10.00,965.82,NaN
362,2024-12-28,181.92,241.71,40.59,58.30,64.43,55.93,7.00,2.21,34.42,...,NaN,14.81,96.00,0.82,312.17,0.0,0.0,50.42,989.92,NaN
363,2024-12-29,100.64,152.54,12.34,38.45,30.46,42.38,7.11,1.19,73.29,...,NaN,14.39,88.29,2.27,284.86,0.0,0.0,87.16,984.90,NaN
364,2024-12-30,111.21,172.67,17.14,45.67,38.27,36.74,8.77,1.30,83.82,...,NaN,12.83,83.15,1.92,247.35,0.0,0.0,69.03,979.98,NaN


In [5]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 21)


In [6]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO (µg/m³)         0
NO2 (µg/m³)        0
NOx (ppb)          0
NH3 (µg/m³)        0
SO2 (µg/m³)        0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
Toluene (µg/m³)    0
AT (°C)            0
RH (%)             0
WS (m/s)           0
WD (deg)           0
RF (mm)            0
TOT-RF (mm)        0
SR (W/mt2)         0
BP (mmHg)          0
dtype: int64


In [7]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [8]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 20)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         199.22        336.47       15.02        60.92   
1  2024-01-02         193.07        329.67       23.40        62.58   
2  2024-01-03         225.34        351.88       32.58        66.10   
3  2024-01-04         259.58        415.72       48.20        74.59   
4  2024-01-05         194.77        333.77       53.25       114.85   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      75.95        29.70         9.35        1.02           8.79   
1      85.93        25.21        12.11        1.21           8.84   
2      98.89        32.93        12.18        1.75           3.35   
3      95.02        26.08        19.48        1.03           3.44   
4     104.38        21.22        20.20        1.14           5.08   

   Benzene (µg/m³)  Toluene (µg/m³)  AT (°C)  RH (%)  WS (m/s)  WD (deg)  \
0             0.06             0.14    10.39   77.89      1

In [9]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [10]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg)
0,2024-01-01,1.120082,0.654759,-0.300220,-0.100525,0.734115,-0.770583,-0.643474,-0.557198,-1.286992,-1.616098,-1.901163,-1.732364,1.140058,0.747895,0.041550,0.0,0.0,-1.609605,-0.135739
1,2024-01-02,1.041810,0.598025,0.300906,-0.052083,1.047721,-1.089218,-0.347049,-0.205874,-1.284720,-1.601511,-1.901163,-1.752266,0.937568,0.667209,0.041550,0.0,0.0,-1.131045,-0.122648
2,2024-01-03,1.452517,0.783327,0.959417,0.050637,1.454970,-0.541364,-0.339531,0.792627,-1.534262,-1.584492,-1.896975,-1.820167,1.711794,0.485663,0.041550,0.0,0.0,-1.874316,-0.108366
3,2024-01-04,1.888296,1.315956,2.079892,0.298390,1.333361,-1.027478,0.444490,-0.538708,-1.530171,-1.545593,-1.811119,-1.801436,1.684249,0.183088,0.041550,0.0,0.0,-2.243001,-0.121458
4,2024-01-05,1.063446,0.632232,2.442145,1.473250,1.627485,-1.372370,0.521818,-0.335309,-1.455627,-1.538299,-1.821589,-1.724169,1.916517,0.727724,-1.139890,0.0,0.0,-2.392498,-0.140499
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,1.971532,0.868261,0.596447,0.713356,0.534575,-0.105991,-0.914123,2.216416,0.665256,0.537973,-0.609138,-1.367101,1.126658,0.445320,2.353326,0.0,0.0,-2.429592,-1.616233
362,2024-12-28,0.899901,-0.135841,1.534001,-0.176981,0.372116,1.090843,-0.895864,1.643202,-0.122007,0.124664,0.033734,-1.214909,2.488252,0.062058,0.041550,0.0,0.0,-1.293750,1.251927
363,2024-12-29,-0.134566,-0.879803,-0.492465,-0.756240,-0.695341,0.129260,-0.884050,-0.242855,1.644790,-0.767598,-1.027947,-1.264079,1.914283,2.986952,0.041550,0.0,0.0,-0.261320,0.654493
364,2024-12-30,-0.000039,-0.711855,-0.148145,-0.545548,-0.449923,-0.270986,-0.705766,-0.039457,2.123420,-0.804067,-0.732686,-1.446710,1.531637,2.280943,0.455573,0.0,0.0,-0.770790,0.068960


In [12]:
df.to_excel("wazirpur2024.xlsx",index=False)